# Quantum State Preparation: 1–8 Qubits

**Qiskit | Ideal, shot-based simulation | Reproducible examples**

This notebook builds and measures:

- **1 qubit:** an equal superposition, $(|0\rangle + |1\rangle)/\sqrt{2}$.
- **2 qubits:** the Bell state, $(|00\rangle + |11\rangle)/\sqrt{2}$.
- **3–8 qubits:** GHZ states, $(|0\rangle^{\otimes n}+|1\rangle^{\otimes n})/\sqrt{2}$.

Each circuit begins in $|0\cdots0\rangle$, uses one Hadamard gate and (for $n\geq2$) a chain of controlled-NOT gates, and ends with computational-basis measurements. You will see the circuit diagram, measurement counts and a histogram.

**Important:** This notebook uses `StatevectorSampler` for ideal statevector-based sampling. It does **not** run on IBM hardware, simulate device noise, or implement quantum error correction. With finite shots, the `0...0` and `1...1` counts fluctuate around 50% each.

**Setup:** From the repository root, install `pip install -r requirements.txt`. In a standalone Colab session, install `qiskit[visualization]==2.5.2` and `matplotlib` first. Then run all cells in order.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorSampler
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram
from IPython.display import display
import numpy as np

SHOTS = 1024
BASE_SEED = 2026

def prepare_superposition_or_ghz(num_qubits: int) -> QuantumCircuit:
    """Return a circuit preparing |+> for n=1 or a GHZ state for n>=2."""
    if not isinstance(num_qubits, int) or not 1 <= num_qubits <= 8:
        raise ValueError("num_qubits must be an integer from 1 to 8")
    circuit = QuantumCircuit(num_qubits)
    circuit.h(0)
    for target in range(1, num_qubits):
        circuit.cx(target - 1, target)
    return circuit


def run_ideal_experiment(num_qubits: int):
    """Prepare, verify, sample and visualize a 1–8 qubit state."""
    prepared = prepare_superposition_or_ghz(num_qubits)
    state = Statevector.from_instruction(prepared)
    expected = np.zeros(2**num_qubits, dtype=complex)
    expected[0] = expected[-1] = 1 / np.sqrt(2)
    assert np.allclose(state.data, expected), "Unexpected prepared state"

    measured = prepared.copy()
    measured.measure_all()  # creates classical register named 'meas'
    sampler = StatevectorSampler(seed=BASE_SEED + num_qubits)
    counts = sampler.run([measured], shots=SHOTS).result()[0].data.meas.get_counts()

    valid_states = {'0' * num_qubits, '1' * num_qubits}
    assert set(counts).issubset(valid_states), "Unexpected ideal measurement outcome"
    assert sum(counts.values()) == SHOTS, "Shot total differs from requested number"

    print(f"{num_qubits}-qubit {'superposition' if num_qubits == 1 else 'GHZ / Bell' if num_qubits == 2 else 'GHZ'} circuit")
    print("Circuit depth:", measured.depth())
    print("Measurement counts:", counts)
    display(measured.draw(output='mpl', style='iqp', fold=100))
    display(plot_histogram(counts, title=f"{num_qubits}-qubit ideal outcomes ({SHOTS} shots)"))
    return measured, counts


## 1 qubit: Hadamard superposition

The ideal measurement outcomes are `0` and `1`, with probabilities 0.5 each.

In [ ]:
circuit_1, counts_1 = run_ideal_experiment(1)

## 2 qubits: Bell-state entanglement

The ideal measurement outcomes are `00` and `11`, with probabilities 0.5 each.

In [ ]:
circuit_2, counts_2 = run_ideal_experiment(2)

## 3 qubits: GHZ-state entanglement

The ideal measurement outcomes are `000` and `111`, with probabilities 0.5 each.

In [ ]:
circuit_3, counts_3 = run_ideal_experiment(3)

## 4 qubits: GHZ-state entanglement

The ideal measurement outcomes are `0000` and `1111`, with probabilities 0.5 each.

In [ ]:
circuit_4, counts_4 = run_ideal_experiment(4)

## 5 qubits: GHZ-state entanglement

The ideal measurement outcomes are `00000` and `11111`, with probabilities 0.5 each.

In [ ]:
circuit_5, counts_5 = run_ideal_experiment(5)

## 6 qubits: GHZ-state entanglement

The ideal measurement outcomes are `000000` and `111111`, with probabilities 0.5 each.

In [ ]:
circuit_6, counts_6 = run_ideal_experiment(6)

## 7 qubits: GHZ-state entanglement

The ideal measurement outcomes are `0000000` and `1111111`, with probabilities 0.5 each.

In [ ]:
circuit_7, counts_7 = run_ideal_experiment(7)

## 8 qubits: GHZ-state entanglement

The ideal measurement outcomes are `00000000` and `11111111`, with probabilities 0.5 each.

In [ ]:
circuit_8, counts_8 = run_ideal_experiment(8)

## Interpretation

The two possible measurement bitstrings arise from the coherent superposition of $|0\cdots0\rangle$ and $|1\cdots1\rangle$ before measurement. A histogram of counts alone does **not** certify entanglement: a classical mixture could show the same distribution. The statevector check above validates the intended *ideal simulated state*, not a hardware entanglement witness.

**Next:** Explore the companion notebook of seeded random circuits, which can generate non-GHZ measurement distributions.